# OneVoice V2 — MT benchmark
Clone source from GitHub, cache EnViT5 on Drive and persist every benchmark report on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
GITHUB_REPO = 'https://github.com/Platypus27-coder/OneVoice.git'
BRANCH = 'main'
REPO = Path('/content/OneVoice')
DRIVE_ROOT = Path('/content/drive/MyDrive/OneVoice')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, GITHUB_REPO, str(REPO)], check=True)
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'model_cache/huggingface')
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'PyYAML', 'torch', 'transformers>=4.45.0', 'sentencepiece', 'sacremoses'], check=True)
REPORT_ROOT = DRIVE_ROOT / 'reports/mt'
print('Source:', REPO, '| Reports:', REPORT_ROOT)


In [ ]:
for direction in ('vi2en', 'en2vi'):
    for suite in ('test', 'minimal', 'safety'):
        for mode in ('raw', 'context'):
            report_dir = REPORT_ROOT / direction / suite / mode
            command = [sys.executable, 'scripts/benchmark_mt_v2.py', '--direction', direction, '--suite', suite, '--report-dir', str(report_dir)]
            if mode == 'context': command.append('--with-context')
            subprocess.run(command, check=True)


In [ ]:
import json
{f'{direction}/{suite}/{mode}': json.loads((REPORT_ROOT / direction / suite / mode / 'aggregate.json').read_text(encoding='utf-8')) for direction in ('vi2en','en2vi') for suite in ('test','minimal','safety') for mode in ('raw','context')}
